# Turkish Lip-Reading — Full Pipeline Runner (Colab, GPU)

Runs **both pipelines** over `video_links.csv`, with all outputs written to
**Google Drive**. Whisper runs on the Colab **GPU** (set Runtime → Change runtime
type → GPU).

For each row, dispatched by the `pipeline` column:

| pipeline | what happens | outputs (on Drive) |
|---|---|---|
| **1** | download video → Whisper ASR | `raw_videos/`, `transcripts/<video>.json` |
| **2** | download → keep solo-reporter scenes → clip → Whisper per clip | `clips/<reporter>/`, `reporter_transcripts/`, appended to `reporter_clips.csv` |

**Resumable via the `completed` column.** When a row finishes it is set to
`completed = 1` and the CSV **is written back to Drive immediately**. Re-running
skips completed rows, so Colab timeouts are harmless — just run again.

### One-time setup in Google Drive
Create `MyDrive/turkish_lipreading/` containing:
```
turkish_lipreading/
├── video_links.csv              ← your latest CSV (with the `completed` column)
├── cookies.txt                  ← OPTIONAL but recommended (see Troubleshooting)
└── pipeline2_reporter_pictures/ ← reference photos, one <slug>.jpeg per reporter
    ├── fatih_portakal.jpeg
    ├── kubra_par.jpeg
    ├── gulsah_ekinci.jpeg
    ├── serdar_cebe.jpeg
    ├── ece_uner.jpeg
    ├── ismail_kucukkaya.jpeg
    ├── cuneyt_ozdemir.jpeg
    ├── cem_ogretir.jpeg
    └── can_okanar.jpeg
```
Everything else (`raw_videos/`, `transcripts/`, `clips/`, `reporter_transcripts/`,
`reporter_clips.csv`) is created automatically.

## 1. Install dependencies
Same stack as local + Whisper. ~2 min.

In [17]:
import os
# Set BEFORE importing torch so the CUDA caching allocator picks it up. Prevents the
# "CUDA out of memory ... try setting expandable_segments:True" failure that large-v3
# hits on a 15 GB T4 when memory gets fragmented across many videos.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install -q dlib-bin
!pip install -q --no-deps face_recognition
!pip install -q "git+https://github.com/ageitgey/face_recognition_models" 2>/dev/null
!pip install -q "setuptools<81"
!pip install -q "scenedetect[opencv]" imageio-ffmpeg
# yt-dlp NIGHTLY: YouTube breaks stable yt-dlp often; nightly usually has the fix.
!pip install -q -U --pre "yt-dlp[default]"
!pip install -q -U openai-whisper
!apt-get -qq install -y ffmpeg >/dev/null 2>&1
import subprocess, sys
print("yt-dlp version:", subprocess.run([sys.executable, "-m", "yt_dlp", "--version"],
                                        capture_output=True, text=True).stdout.strip())
import torch
print("Torch CUDA available:", torch.cuda.is_available(),
      "| device:", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"))
print("If 'import face_recognition' fails below: Runtime > Restart session,")
print("then re-run from the Mount Drive cell (skip this install).")

  Preparing metadata (setup.py) ... done
yt-dlp version: 2026.06.13.234541
Torch CUDA available: True | device: Tesla T4
If 'import face_recognition' fails below: Runtime > Restart session,
then re-run from the Mount Drive cell (skip this install).


## 2. Mount Google Drive

In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Config — edit these
Use the filters to chunk work (one reporter, or a few videos each) before a full run.

In [19]:
from pathlib import Path

# ====== EDIT THESE ======
DRIVE_BASE = Path('/content/drive/MyDrive/turkish_lipreading')

WHISPER_MODEL = "large-v3"     # GPU makes this fast; use "small"/"base" for quick checks

# Limit the run (None = no limit)
PIPELINE_FILTER         = None     # None, "1", or "2"  -> run only that pipeline
REPORTERS_FILTER        = None     # e.g. ['fatih_portakal', 'cuneyt_ozdemir'] (pipeline 2)
MAX_ROWS                = 11       # first N rows from the top; None = whole CSV

# Scene / face params (match local pipeline2.py)
SCENE_THRESHOLD = 30.0
# Solo-presenter filter -- mirrors scripts/presenter_filter.py, keep in sync.
SECONDS_PER_SAMPLE = 0.2  # faces counted + yaw measured every 0.2 s (YuNet ~10 ms)
IDENTITY_EVERY  = 10     # dlib identity encoding (~200 ms) every 10th sample + each new track
SCENE_EDGE_TRIM = 0.3    # cut transitions (glitch wipes, dissolves) off both scene ends
ANALYSIS_MAX_HEIGHT = 1080
DETECT_WIDTH    = 960
YUNET_MIN_SCORE = 0.5
FACE_SCORE_MIN  = 0.80   # below this YuNet hits are decor: tree 0.69, figurine 0.60, bust 0.61
MIN_FACE_FRACTION = 0.045  # a face >= 4.5% of frame height is a PERSON ON SET: split-screen
                           # guests (~9%) and interpreter boxes (~6%) disqualify the frame
TRACK_MAX_SHIFT = 0.5    # identity is voted per track (same face, same place)
MAX_FACE_YAW    = 0.30   # "faces the camera directly": 0 = at the lens, 1 = full profile
FACE_TOLERANCE  = 0.6    # median track distance; dlib's same-person threshold
MIN_DURATION    = 2.0    # word clips need +-0.6 s of context; shorter clips are useless
YUNET_MODEL_URL = ("https://github.com/opencv/opencv_zoo/raw/main/models/"
                   "face_detection_yunet/face_detection_yunet_2023mar.onnx")
DOWNLOAD_FORMAT = "bestvideo[height>=720]+bestaudio/best/best"

# --- YouTube access (Colab IPs often hit the bot check; see Troubleshooting cell) ---
COOKIES_FILE = DRIVE_BASE / 'cookies.txt'   # used automatically IF it exists
# Each video is retried with these player clients in order until one works. Different
# clients survive the bot check on different days, so a list is far more reliable than
# a single client. Reorder/trim if you find one that consistently works for you.
YT_CLIENT_FALLBACKS = ["default", "tv", "mweb", "web_safari", "android"]

KEEP_P2_SOURCE_VIDEO = False   # False = delete the big source video after clipping (saves Drive)
# ========================

CSV_PATH      = DRIVE_BASE / 'video_links.csv'
REF_DIR       = DRIVE_BASE / 'pipeline2_reporter_pictures'
RAW_DIR       = DRIVE_BASE / 'raw_videos'            # pipeline 1 full videos
TRANSCRIPTS   = DRIVE_BASE / 'transcripts'           # pipeline 1 transcripts
P2_RAW        = DRIVE_BASE / 'pipeline2_raw'         # pipeline 2 source videos (transient unless KEEP_P2_SOURCE_VIDEO)
CLIPS_DIR     = DRIVE_BASE / 'clips'                 # pipeline 2 clips
P2_TRANSCRIPTS= DRIVE_BASE / 'reporter_transcripts'  # pipeline 2 per-clip transcripts
REPORTER_CSV  = DRIVE_BASE / 'reporter_clips.csv'    # pipeline 2 manifest + transcript text

for d in (RAW_DIR, TRANSCRIPTS, P2_RAW, CLIPS_DIR, P2_TRANSCRIPTS):
    d.mkdir(parents=True, exist_ok=True)

print("CSV exists:", CSV_PATH.exists(), "->", CSV_PATH)
print("Reference dir exists:", REF_DIR.exists(), "->", REF_DIR)
print("cookies.txt present:", COOKIES_FILE.exists())
if REF_DIR.exists():
    print("Reference images:", sorted(p.name for p in REF_DIR.iterdir()))

CSV exists: True -> /content/drive/MyDrive/turkish_lipreading/video_links.csv
Reference dir exists: True -> /content/drive/MyDrive/turkish_lipreading/pipeline2_reporter_pictures
cookies.txt present: False
Reference images: ['cuneyt_ozdemir.jpeg', 'ece_uner.jpeg', 'fatih_portakal.jpeg', 'gulsah_ekinci.jpeg', 'ismail_kucukkaya.jpeg', 'kubra_par.jpeg', 'serdar_cebe.jpeg']


## ⚠️ Troubleshooting: downloads fail / "Sign in to confirm you're not a bot"
YouTube blocks most **Colab/datacenter IPs**, so `yt-dlp` returns no file. The notebook
now (a) uses **yt-dlp nightly**, (b) **retries every video across several player
clients** (`YT_CLIENT_FALLBACKS`), and (c) uses your `cookies.txt` if present. That
handles most cases. When it still fails, it's almost always the cookie.

### Why cookies "sometimes" stop working — and how to export them correctly
YouTube **rotates/invalidates** the session cookie if the browser you exported from keeps
using YouTube (or you log out). That's why a cookie works for a while then dies. Export
them so they can't be rotated:

1. Use a **throwaway Google account**, not your main one (this can get the account
   rate-limited).
2. Open a **private / incognito window** and log into `youtube.com`.
3. With an extension like *Get cookies.txt LOCALLY*, export `cookies.txt` while on
   `youtube.com`.
4. **Close the incognito window WITHOUT logging out.** Closing incognito ends the
   session locally but leaves the cookie valid; logging out or normal-window browsing
   is what invalidates it.
5. Upload that `cookies.txt` to `MyDrive/turkish_lipreading/`. Re-export when it expires
   (every week or two).

### If it *still* fails on Colab
Colab's datacenter IP is the hard part. Options, in order of effort:
- **Re-run later / different Colab session** — you sometimes get a less-blocked IP.
- **Run the diagnostic cell** to see which client (if any) works for a given URL, then
  put that client first in `YT_CLIENT_FALLBACKS`.
- **PO-token provider** (heavy but most reliable): install the
  `bgutil-ytdlp-pot-provider` plugin so yt-dlp can mint the proof-of-origin tokens
  YouTube now demands from datacenter IPs.
- **Download locally, process on Colab** — the bottleneck Colab gives you is the *GPU*
  (Whisper) and CPU face filtering, not the download. Download the raw videos on your
  home machine (your IP isn't blocked), drop them in Drive, and point the notebook at
  those files instead of re-downloading. This sidesteps the cookie problem entirely.

### Diagnostic (optional) — see the real yt-dlp error for one URL
Run this after the config + core-function cells if downloads fail.

In [20]:
def diagnose(url):
    """Try each player client for one URL and show which (if any) can list formats."""
    import subprocess, sys
    for client in (YT_CLIENT_FALLBACKS or [None]):
        cmd = [sys.executable, "-m", "yt_dlp", "-F", "--no-playlist"]
        if COOKIES_FILE.exists():
            cmd += ["--cookies", str(COOKIES_FILE)]
        if client:
            cmd += ["--extractor-args", f"youtube:player_client={client}"]
        cmd += [url]
        res = subprocess.run(cmd, capture_output=True, text=True)
        ok = "format code" in res.stdout.lower() or "\nID " in res.stdout
        print(f"=== client={client!r}: {'OK' if ok else 'FAIL'} ===")
        if not ok:
            err = (res.stderr or res.stdout).strip().splitlines()
            print("   " + "\n   ".join(err[-4:]))
        else:
            print("   (formats listed — this client works)")

# diagnose("https://www.youtube.com/watch?v=QgKPn7DZrgQ")  # uncomment to test

## 4. Core functions
Self-contained copies of the project's pipeline logic (download, scene detect,
solo-face filter, clip export) plus a Whisper transcriber that writes the same
JSON schema as `scripts/run_whisper.py`.

In [21]:
import math, re, statistics, subprocess, sys, json, unicodedata, time
import cv2
import numpy as np
import face_recognition
from scenedetect import SceneManager, open_video
from scenedetect.detectors import ContentDetector

def yt_dlp_base_cmd():
    return [sys.executable, "-m", "yt_dlp"]

def extract_video_id(url):
    for pat in (r"youtu\.be/([A-Za-z0-9_-]{11})",
                r"[?&]v=([A-Za-z0-9_-]{11})",
                r"/shorts/([A-Za-z0-9_-]{11})"):
        m = re.search(pat, url)
        if m:
            return m.group(1)
    return None

def normalize_text(text):
    # strip: Whisper word tokens carry a leading space; " Yargi" and "Yargi" must
    # not become two word classes. Mirrors run_whisper.normalize_text.
    return unicodedata.normalize("NFC", text).strip()

def safe_name(raw):
    s = re.sub(r"[^A-Za-z0-9._-]+", "_", raw).strip("._")
    return s or "video"

def _found_video(downloads_dir, stem):
    for p in downloads_dir.glob(f"{stem}.*"):
        if p.suffix.lower() in (".mp4", ".mkv", ".webm", ".mov"):
            return p
    return None

def _cleanup_partials(downloads_dir, stem):
    # remove .part/.ytdl/failed remnants so the next client attempt starts clean
    for p in downloads_dir.glob(f"{stem}.*"):
        try:
            p.unlink()
        except Exception:
            pass

def _ytdlp_once(url, downloads_dir, stem, start, end, client):
    out_template = str(downloads_dir / f"{stem}.%(ext)s")
    cmd = [*yt_dlp_base_cmd(), "-f", DOWNLOAD_FORMAT,
           "--merge-output-format", "mp4",
           "--retries", "5", "--fragment-retries", "10", "--retry-sleep", "5",
           "--no-playlist", "-o", out_template]
    if COOKIES_FILE and Path(COOKIES_FILE).exists():
        cmd += ["--cookies", str(COOKIES_FILE)]
    if client:
        cmd += ["--extractor-args", f"youtube:player_client={client}"]
    if start and end:
        cmd += ["--download-sections", f"*{start}-{end}", "--force-keyframes-at-cuts"]
    cmd += [url]
    return subprocess.run(cmd, capture_output=True, text=True)

def download_video(url, downloads_dir, start=None, end=None):
    downloads_dir.mkdir(parents=True, exist_ok=True)
    vid = extract_video_id(url) or "video"
    # Unique stem per section, so the same url with different start/end never collides.
    stem = f"{vid}_{safe_name(start)}-{safe_name(end)}" if (start and end) else vid

    existing = _found_video(downloads_dir, stem)
    if existing:                      # resume: don't re-download what's already here
        return existing

    last_err = ""
    for client in (YT_CLIENT_FALLBACKS or [None]):
        res = _ytdlp_once(url, downloads_dir, stem, start, end, client)
        got = _found_video(downloads_dir, stem)
        if got:
            return got
        last_err = (res.stderr or res.stdout or "").strip()
        _cleanup_partials(downloads_dir, stem)
        time.sleep(2)                 # small backoff between client attempts

    lines = last_err.splitlines()
    tail = "\n      ".join(lines[-6:]) if lines else "(no yt-dlp output)"
    hint = ""
    if "Sign in to confirm" in last_err or "bot" in last_err.lower():
        hint = ("\n    -> This is the YouTube bot check. Re-export a FRESH cookies.txt "
                "(see the Troubleshooting cell) — cookies expire/rotate.")
    raise RuntimeError(
        f"download failed for {url} (tried clients {YT_CLIENT_FALLBACKS})\n"
        f"    yt-dlp said:\n      {tail}{hint}")

def load_reference_encoding(reference_image):
    # LARGEST face, not encodings[0]: HOG also finds faces in background art and
    # detection order is not size order. Mirrors presenter_filter.
    image = face_recognition.load_image_file(str(reference_image))
    locs = face_recognition.face_locations(image, model="hog")
    if not locs:
        raise RuntimeError(f"No face found in reference image: {reference_image}")
    locs.sort(key=lambda b: b[2] - b[0], reverse=True)
    encs = face_recognition.face_encodings(image, known_face_locations=locs[:1])
    if not encs:
        raise RuntimeError(f"Face not encodable in reference image: {reference_image}")
    return encs[0]

def _tc_seconds(tc):
    # scenedetect 0.7 deprecates get_seconds() in favour of the `seconds` property.
    return tc.seconds if hasattr(tc, "seconds") else tc.get_seconds()

def video_duration_sec(video_path):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 0
    frames = cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0
    cap.release()
    return (frames / fps) if fps else 0.0

def detect_scenes(video_path, threshold):
    video = open_video(str(video_path))
    mgr = SceneManager()
    mgr.add_detector(ContentDetector(threshold=threshold))
    mgr.detect_scenes(video=video, show_progress=False)
    scenes = []
    for s_tc, e_tc in mgr.get_scene_list():
        a, b = _tc_seconds(s_tc), _tc_seconds(e_tc)
        if b > a:
            scenes.append((a, b))
    if not scenes:
        # PySceneDetect returns [] for a single continuous shot (no cuts). Without this
        # fallback every cut-free talking-head video yielded 0 clips. Treat the whole
        # video as one scene so it still goes through the solo-face filter.
        dur = video_duration_sec(video_path)
        if dur > 0:
            scenes = [(0.0, dur)]
    return scenes

def sample_frames(video_path, step=SECONDS_PER_SAMPLE):
    """Yield (t, frame_bgr, source_height) every `step` s in one sequential ffmpeg pass."""
    cap = cv2.VideoCapture(str(video_path))
    src_w, src_h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    if src_w <= 0 or src_h <= 0:
        raise RuntimeError(f"Cannot read frame size of {video_path}")
    sc = min(1.0, ANALYSIS_MAX_HEIGHT / src_h)
    w, h = max(2, int(round(src_w * sc / 2)) * 2), max(2, int(round(src_h * sc / 2)) * 2)
    proc = subprocess.Popen(["ffmpeg", "-v", "error", "-nostdin", "-i", str(video_path),
                             "-an", "-sn", "-vf", f"fps={1.0 / step:g}:round=near,scale={w}:{h}",
                             "-f", "rawvideo", "-pix_fmt", "bgr24", "-"],
                            stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
    size, k = w * h * 3, 0
    try:
        while True:
            buf = proc.stdout.read(size)
            if len(buf) < size:
                break
            yield k * step, np.frombuffer(buf, np.uint8).reshape(h, w, 3), src_h
            k += 1
    finally:
        proc.stdout.close(); proc.kill(); proc.wait()

_detector = None
def yunet_detector():
    global _detector
    if _detector is None:
        model = Path("/content/face_detection_yunet_2023mar.onnx")
        if not model.exists():
            import urllib.request
            urllib.request.urlretrieve(YUNET_MODEL_URL, model)
        _detector = cv2.FaceDetectorYN.create(str(model), "", (320, 320),
                                              score_threshold=YUNET_MIN_SCORE)
    return _detector

def detect_faces(frame_bgr):
    """YuNet detections as (score, x, y, w, h) in frame pixels, largest first."""
    fh, fw = frame_bgr.shape[:2]
    s = min(1.0, DETECT_WIDTH / fw)
    small = cv2.resize(frame_bgr, (int(fw * s), int(fh * s))) if s < 1.0 else frame_bgr
    det = yunet_detector()
    det.setInputSize((small.shape[1], small.shape[0]))
    _, raw = det.detect(small)
    faces = [] if raw is None else [(float(r[14]), float(r[0]) / s, float(r[1]) / s,
                                     float(r[2]) / s, float(r[3]) / s) for r in raw]
    faces.sort(key=lambda f: f[4], reverse=True)
    return faces

def _dlib_box(face, shape):
    """dlib (top, right, bottom, left): the bottom-anchored square of YuNet's width."""
    _, x, y, w, h = face
    fh, fw = shape[:2]
    return (int(max(0, y + h - w)), int(min(fw - 1, x + w)), int(min(fh - 1, y + h)), int(max(0, x)))

def _shift(a, b):
    ax, ay, bx, by = a[1] + a[3] / 2, a[2] + a[4] / 2, b[1] + b[3] / 2, b[2] + b[4] / 2
    return math.hypot(ax - bx, ay - by) / max(1.0, min(a[3], b[3]))

def face_yaw(rgb, box):
    """Lateral head turn: 0.0 facing the camera, 1.0 full profile.

    Where the nose tip sits horizontally inside the jaw outline. None when
    landmarks cannot be fitted -- callers must treat that as a failure, since an
    unmeasurable face is not a verified frontal one. Mirrors presenter_filter.
    """
    marks = face_recognition.face_landmarks(rgb, [box])
    if not marks:
        return None
    chin, nose = marks[0].get("chin"), marks[0].get("nose_tip")
    if not chin or not nose:
        return None
    nose_x = sum(p[0] for p in nose) / len(nose)
    dl, dr = abs(nose_x - chin[0][0]), abs(chin[-1][0] - nose_x)
    if dl + dr <= 0:
        return 1.0
    return abs(dr - dl) / (dr + dl)

def analyze_frames(frames, target_enc):
    """Per-sample face records. Mirrors presenter_filter.analyze_frames."""
    records, prev = [], None
    for i, (t, frame, src_h) in enumerate(frames):
        fh, fw = frame.shape[:2]
        faces = [f for f in detect_faces(frame) if f[4] >= 0.03 * fh]
        jumped = prev is None or len(faces) != len(prev) or (
            faces and _shift(prev[0], faces[0]) >= TRACK_MAX_SHIFT)
        measure = i % IDENTITY_EVERY == 0 or jumped
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) if faces else None
        out = []
        for f in faces:
            box = _dlib_box(f, frame.shape)
            yaw = face_yaw(rgb, box)
            dist = None
            if measure:
                enc = face_recognition.face_encodings(rgb, known_face_locations=[box])
                if enc:
                    dist = float(face_recognition.face_distance([target_enc], enc[0])[0])
            out.append([f[0], f[1] / fw, f[2] / fh, f[3] / fw, f[4] / fh, dist, yaw])
        records.append({"t": t, "src_h": src_h, "frame_h": fh, "frame_w": fw, "faces": out})
        prev = faces
    return records

def solo_segments(records, start, end, tol, min_duration=MIN_DURATION):
    """Sub-ranges of one scene where only the target reporter is on screen.

    A 0.2 s sample passes when exactly one detection is a person on set (score >=
    FACE_SCORE_MIN, >= MIN_FACE_FRACTION of frame height), its track (same face, no
    jump) has median identity distance <= tol, and its own yaw <= MAX_FACE_YAW.
    SCENE_EDGE_TRIM comes off both scene ends. Mirrors presenter_filter.solo_segments.
    """
    lo, hi = start + SCENE_EDGE_TRIM, end - SCENE_EDGE_TRIM
    recs = [r for r in records if lo <= r["t"] <= hi]
    if hi - lo < min_duration or not recs:
        return []
    solo = []
    for r in recs:
        on = [f for f in r["faces"] if f[0] >= FACE_SCORE_MIN and f[4] >= MIN_FACE_FRACTION]
        solo.append(on[0] if len(on) == 1 else None)
    def px(r, f):
        return (f[0], f[1] * r["frame_w"], f[2] * r["frame_h"], f[3] * r["frame_w"], f[4] * r["frame_h"])
    good, i = [False] * len(recs), 0
    while i < len(recs):
        if solo[i] is None:
            i += 1; continue
        j = i
        while (j + 1 < len(recs) and solo[j + 1] is not None
               and _shift(px(recs[j], solo[j]), px(recs[j + 1], solo[j + 1])) < TRACK_MAX_SHIFT):
            j += 1
        dists = [solo[k][5] for k in range(i, j + 1) if solo[k][5] is not None]
        if dists and statistics.median(dists) <= tol:
            for k in range(i, j + 1):
                good[k] = solo[k][6] is not None and solo[k][6] <= MAX_FACE_YAW
        i = j + 1
    half, out, k = SECONDS_PER_SAMPLE / 2.0, [], 0
    while k < len(recs):
        if not good[k]:
            k += 1; continue
        m = k
        while m + 1 < len(recs) and good[m + 1]:
            m += 1
        a, b = max(lo, recs[k]["t"] - half), min(hi, recs[m]["t"] + half)
        if b - a >= min_duration:
            out.append((a, b, int(statistics.median(solo[q][4] * recs[q]["src_h"]
                                                    for q in range(k, m + 1)))))
        k = m + 1
    return out

def export_clip(input_video, output_video, start, end):
    output_video.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
                    "-ss", f"{start:.3f}", "-to", f"{end:.3f}", "-i", str(input_video),
                    "-c:v", "libx264", "-preset", "veryfast", "-crf", "22",
                    "-c:a", "aac", str(output_video)], check=True)

# ---- Whisper (loaded once, on GPU if available) ----
_model = None
def get_model():
    global _model
    if _model is None:
        import whisper
        print(f"Loading Whisper '{WHISPER_MODEL}' ...")
        _model = whisper.load_model(WHISPER_MODEL)
    return _model

def transcribe(media_path, out_json):
    """Transcribe a video/clip; write run_whisper-style JSON; return plain text."""
    model = get_model()
    result = model.transcribe(str(media_path), word_timestamps=True,
                              language="tr", verbose=False)
    data = {"video": Path(media_path).name, "segments": []}
    texts = []
    for seg in result.get("segments", []):
        sd = {"start_sec": round(seg["start"], 3), "end_sec": round(seg["end"], 3),
              "text": normalize_text(seg["text"]), "words": []}
        texts.append(sd["text"])
        for w in seg.get("words", []):
            surface = normalize_text(w["word"])
            conf = round(w.get("probability", 1.0), 6)
            sd["words"].append({"surface_form": surface, "lemma": surface,
                                "start_sec": round(w["start"], 3), "end_sec": round(w["end"], 3),
                                "confidence": conf, "manual_review": int(conf < 0.7)})
        data["segments"].append(sd)
    out_json.parent.mkdir(parents=True, exist_ok=True)
    out_json.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    return " ".join(texts).strip()

print("Core functions loaded.")

Core functions loaded.


## 5. CSV + completion helpers
Reads `video_links.csv` from Drive, adds the `completed` column if missing, and
writes the whole CSV back to Drive each time a row finishes.

In [22]:
import csv, os

LINK_FIELDS = ["url", "channel", "start", "end", "pipeline", "reporter", "completed"]
CLIP_FIELDS = ["reporter", "source_url", "source_video", "clip_file",
               "start_sec", "end_sec", "duration_sec", "face_height_px",
               "transcript_text", "transcript_json"]

def col(row, key):
    return (row.get(key) or "").strip()

def load_rows():
    with open(CSV_PATH, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    for r in rows:
        r.setdefault("completed", "")   # tolerate a CSV without the column yet
    return rows

def write_all_rows(rows):
    with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=LINK_FIELDS)
        w.writeheader()
        for r in rows:
            w.writerow({k: (r.get(k) or "") for k in LINK_FIELDS})

def mark_completed(rows, row):
    row["completed"] = "1"
    write_all_rows(rows)

def append_reporter_clips(clip_rows):
    if not clip_rows:
        return
    is_new = not REPORTER_CSV.exists()
    with open(REPORTER_CSV, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=CLIP_FIELDS)
        if is_new:
            w.writeheader()
        for r in clip_rows:
            w.writerow(r)

def find_reference_image(reporter):
    for ext in (".jpeg", ".jpg", ".png", ".webp"):
        p = REF_DIR / f"{reporter}{ext}"
        if p.exists():
            return p
    return None

print("CSV/completion helpers loaded.")

CSV/completion helpers loaded.


## 6. Run
Selects the **first `MAX_ROWS` rows from the top** of `video_links.csv` (after the
`PIPELINE_FILTER` / `REPORTERS_FILTER`). `MAX_ROWS` counts the same rows every run,
whether or not they're done — already-completed rows just print *"already processed,
skipping"* instead of being re-run. Set `MAX_ROWS = None` for the full dataset.

Outputs go to Drive; the CSV is rewritten to Drive after every newly completed row
(interrupt-safe resume). GPU memory is freed between videos.

In [23]:
import gc

def free_gpu():
    """Release GPU memory between videos to avoid fragmentation OOM over a long run."""
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass

def process_pipeline1(row, rows):
    url, channel = col(row, "url"), col(row, "channel")
    start, end = col(row, "start"), col(row, "end")
    print(f"[P1] {channel} {url} {start}-{end if end else ''}")
    vpath = download_video(url, RAW_DIR, start or None, end or None)
    out_json = TRANSCRIPTS / (vpath.stem + ".json")
    if not out_json.exists():
        transcribe(vpath, out_json)
    mark_completed(rows, row)
    print(f"   -> transcript {out_json.name} (completed)")

_enc_cache = {}
def process_pipeline2(row, rows):
    reporter, url = col(row, "reporter"), col(row, "url")
    ref = find_reference_image(reporter)
    if not ref:
        print(f"[P2] skip {reporter}: no reference image in {REF_DIR}")
        return
    if reporter not in _enc_cache:
        _enc_cache[reporter] = load_reference_encoding(ref)
    enc = _enc_cache[reporter]
    print(f"[P2] {reporter} {url}")
    vpath = download_video(url, P2_RAW)
    scenes = detect_scenes(vpath, SCENE_THRESHOLD)
    records = analyze_frames(sample_frames(vpath), enc)
    base, clip_rows = safe_name(vpath.stem), []
    for idx, (sc_s, sc_e) in enumerate(scenes, start=1):
        # One scene can yield several clips: solo_segments cuts it at the samples
        # where someone else (or nobody) is on screen instead of dropping the scene.
        for s, e, face_h in solo_segments(records, sc_s, sc_e, FACE_TOLERANCE):
            clip = CLIPS_DIR / reporter / f"{reporter}_{base}_sc{idx:04d}_{s:.2f}-{e:.2f}.mp4"
            export_clip(vpath, clip, s, e)
            txt = transcribe(clip, P2_TRANSCRIPTS / (clip.stem + ".json"))
            clip_rows.append({
                "reporter": reporter, "source_url": url, "source_video": vpath.name,
                "clip_file": str(clip.relative_to(DRIVE_BASE)),
                "start_sec": round(s, 3), "end_sec": round(e, 3),
                "duration_sec": round(e - s, 3),
                "face_height_px": face_h,
                "transcript_text": txt,
                "transcript_json": str((P2_TRANSCRIPTS / (clip.stem + ".json")).relative_to(DRIVE_BASE)),
            })
    if not KEEP_P2_SOURCE_VIDEO:
        try:
            vpath.unlink()
        except Exception:
            pass
    append_reporter_clips(clip_rows)
    mark_completed(rows, row)
    print(f"   -> {len(clip_rows)} clips ({len(scenes)} scenes) (completed)")

def run():
    rows = load_rows()

    # Select rows from the TOP of the CSV (filters first, then MAX_ROWS). Completed
    # rows are kept in the selection so MAX_ROWS always means "the first N rows",
    # independent of how many were already done — they're just skipped below.
    selected = rows
    if PIPELINE_FILTER:
        selected = [r for r in selected if col(r, "pipeline") == PIPELINE_FILTER]
    if REPORTERS_FILTER:
        selected = [r for r in selected
                    if col(r, "pipeline") != "2" or col(r, "reporter") in REPORTERS_FILTER]
    if MAX_ROWS:
        selected = selected[:MAX_ROWS]

    n = len(selected)
    print(f"{n} rows selected (MAX_ROWS={MAX_ROWS})\n")
    done = skipped = failed = 0
    for i, row in enumerate(selected, start=1):
        url = col(row, "url")
        if col(row, "completed") == "1":
            print(f"[{i}/{n}] already processed, skipping: {url}")
            skipped += 1
            continue
        pl = col(row, "pipeline")
        try:
            if pl == "1":
                process_pipeline1(row, rows)
            elif pl == "2":
                process_pipeline2(row, rows)
            else:
                print(f"[{i}/{n}] unknown pipeline '{pl}', skipping: {url}")
                continue
            done += 1
        except Exception as e:
            print(f"[{i}/{n}]    [FAIL] {url}: {e}")
            failed += 1
        finally:
            free_gpu()

    print(f"\nFinished. processed={done}, already-done={skipped}, failed={failed} of {n} selected.")
    print(f"CSV updated on Drive: {CSV_PATH}")

run()

11 rows selected (MAX_ROWS=11)

[1/11] already processed, skipping: https://www.youtube.com/watch?v=QgKPn7DZrgQ
[2/11] already processed, skipping: https://www.youtube.com/watch?v=VNt0zif5GYk
[3/11] already processed, skipping: https://www.youtube.com/watch?v=A4qgy-_ucUY
[P1] HALK TV https://youtube.com/shorts/aL2A_uyv2SE?si=zud8hlJyzNQBNEKA 00:00:00-00:00:39
Loading Whisper 'large-v3' ...


100%|██████████| 3902/3902 [00:09<00:00, 430.90frames/s]


   -> transcript aL2A_uyv2SE_00_00_00-00_00_39.json (completed)
[P1] HALK TV https://youtube.com/shorts/bGJm0ACw0ts?si=wZDZRbO6HtCH1YK7 00:00:00-00:00:16


100%|██████████| 1600/1600 [00:05<00:00, 303.73frames/s]


   -> transcript bGJm0ACw0ts_00_00_00-00_00_16.json (completed)
[P1] NTV https://youtu.be/yLELC9qj9yg?si=YoUjd1jED3zzR9rW 00:00:00-00:00:27


...

   -> 1 clips (1 scenes) (completed)
[P2] fatih_portakal https://youtu.be/uaBe-qSypWE


INFO:pyscenedetect:Detecting scenes...
100%|██████████| 22714/22714 [01:02<00:00, 365.41frames/s]


   -> 1 clips (1 scenes) (completed)
[P2] fatih_portakal https://youtu.be/Iazp8RVj6lc


INFO:pyscenedetect:Detecting scenes...


   -> 0 clips (1 scenes) (completed)

Finished. processed=8, already-done=3, failed=0 of 11 selected.
CSV updated on Drive: /content/drive/MyDrive/turkish_lipreading/video_links.csv


## 7. (Optional) Zip clips for download
Clips already live on Drive; this just bundles them.

In [24]:
import shutil
archive = shutil.make_archive(str(DRIVE_BASE / "clips"), "zip", str(CLIPS_DIR))
print("Created:", archive)
# from google.colab import files; files.download(archive)

Created: /content/drive/MyDrive/turkish_lipreading/clips.zip
